### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [2]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [3]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [4]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [5]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [6]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [7]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [8]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [9]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [10]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [11]:
tfidfvect.vocabulary_['cocoliso']

KeyError: 'cocoliso'

Es muy útil tener el diccionario opuesto que va de índices a términos

In [13]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [14]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [15]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [16]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [17]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

Podemos ver los valores de similaridad ordenados de mayor a menor

In [18]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ], shape=(11314,))

Después vemos a qué documentos corresponden

In [19]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  1534, 10055,  4750], shape=(11314,))

Obtenemos los 5 documentos más similares:

In [20]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [21]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [22]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [23]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None


Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [24]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [25]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


In [ ]:
import numpy as np

# Punto 1. Elegir 5 documentos al azar
np.random.seed(42)  # Para reproducibilidad
random_indices = np.random.choice(len(newsgroups_train.data), 5, replace=False)

for idx in random_indices:
    print("=" * 50)
    print(f"\nIDX {idx}")
    print(f"\nCLASE:\n{newsgroups_train.target_names[y_train[idx]]}")
    print(f"\nDOCUMENTO (200 chars):\n\n {newsgroups_train.data[idx][:200]}...")
    
    # Calcular similitud coseno con todo el set de entrenamiento
    cossim = cosine_similarity(X_train[idx], X_train)[0]
    
    # Obtener los 5 más similares 
    # (asumiendo que el documento mas similar es el mismo y por lo tanto está en la posición 0)
    most_sim_indices = np.argsort(cossim)[::-1][1:6]
    
    print("\n5 CLASES MÁS SIMILARES:\n")
    for sim_idx in most_sim_indices:
        sim_score = cossim[sim_idx]
        sim_label = newsgroups_train.target_names[y_train[sim_idx]]
        print(f" - [Similitud: {sim_score:.3f}] Clase: {sim_label}")
    
    print("\n5 DOCUMENTOS MÁS SIMILARES:\n")
    for sim_idx in most_sim_indices:
        print(f" - [Texto (200):\n\n{newsgroups_train.data[sim_idx][:200]}...]")
        


IDX 7492

CLASE:
comp.sys.mac.hardware

DOCUMENTO (200 chars):

 Could someone please post any info on these systems.

Thanks.
BoB
-- 
---------------------------------------------------------------------- 
Robert Novitskey | "Pursuing women is similar to banging o...

5 CLASES MÁS SIMILARES:

 - [Similitud: 0.667] Clase: comp.sys.mac.hardware
 - [Similitud: 0.348] Clase: comp.sys.ibm.pc.hardware
 - [Similitud: 0.180] Clase: comp.sys.mac.hardware
 - [Similitud: 0.155] Clase: misc.forsale
 - [Similitud: 0.141] Clase: comp.sys.mac.hardware

5 DOCUMENTOS MÁS SIMILARES:

 - [Texto (200):

Hey everybody:

   I want to buy a mac and I want to get a good price...who doesn't?  So,
could anyone out there who has found a really good deal on a Centris 650
send me the price.  I don't want to k...]
 - [Texto (200):

Hay all:

    Has anyone out there heard of any performance stats on the fabled p24t.
 I was wondering what it's performance compared to the 486/66 and/or
pentium would be.  Any info w

Interpretación

La solución anterior está basada en el ejemplo de este mismo documento donde se usa la similitud de coseno con vectorización TF-IDF, efectiva para agrupar documentos con temáticas similares. En muchos casos, los documentos más cercanos pertenecen a la misma clase que la original o en apariencia  "semánticamente" relacionada, aunque en este caso la técnica se la similitud léxica (importancia de las palabras y palabras compartidas) no implica similitud semántica literal.

In [ ]:
# 2. Clasificar el test set usando el vecino más cercano (similitud máxima)
# Nota: Para optimizar, usamos el producto punto de matrices (X_test @ X_train.T)
# que es equivalente a cosine_similarity si los vectores están normalizados (L2).

# TfidfVectorizer por defecto normaliza a L2, así que podemos multiplicar:
dot_product = X_test @ X_train.T

# Obtenemos el índice del documento de train con máxima similitud para cada test
preds_prototype = []
for i in range(dot_product.shape[0]):
    best_match_idx = dot_product[i].argmax()
    preds_prototype.append(y_train[best_match_idx])

# Evaluación
score_proto = f1_score(y_test, preds_prototype, average='macro')
print(f"F1-Score Macro (Clasificador por Prototipos): {score_proto:.4f}")

F1-Score Macro (Clasificador por Prototipos): 0.5050


Interpretación

Interesantes resultados pero inferiores al ejemplo visto antes con el clasificador MultinomialNB por defecto.

Ahora se clasifica correctamente como la mitad de las veces.
Con 20 clases, un 50% es mejor que el azar, es decir que funciona moderadamente bien pero aun es limitado.

In [45]:
# Optimización
import numpy as np
from sklearn.naive_bayes import MultinomialNB, ComplementNB

# Configuraciones de prueba
vectorizers = {
    'TF-IDF': TfidfVectorizer(),
    'TF-IDF Filtro': TfidfVectorizer(min_df=2, max_df=0.8, stop_words='english'),
    'TF-IDF Sublinear': TfidfVectorizer(sublinear_tf=True, stop_words='english'),
    'TF-IDF MaxFeat': TfidfVectorizer(max_features=10000, stop_words='english'),
    'Count': CountVectorizer(stop_words='english'),
    'Count MaxFeat': CountVectorizer(max_features=10000, stop_words='english')
}

alphas = [0.001, 0.01, 0.1, 1.0]
results = []

for vec_name, vec in vectorizers.items():
    X_train_vec = vec.fit_transform(newsgroups_train.data)
    X_test_vec = vec.transform(newsgroups_test.data)
    
    for model_name, model_class in [('MNB', MultinomialNB), ('CNB', ComplementNB)]:
        for alpha in alphas:
            model = model_class(alpha=alpha)
            model.fit(X_train_vec, y_train)
            f1 = f1_score(y_test, model.predict(X_test_vec), average='macro')
            
            results.append((f1, vec_name, model_name, alpha))
            print(f"{vec_name}+{model_name}(α={alpha}): {f1:.4f}")

# Mejor configuración
best = max(results)
print(f"\nMEJOR: {best[1]} en {best[2]} (con α={best[3]})")
print(f"F1-Score Macro: {best[0]:.4f}")

TF-IDF+MNB(α=0.001): 0.6713
TF-IDF+MNB(α=0.01): 0.6829
TF-IDF+MNB(α=0.1): 0.6565
TF-IDF+MNB(α=1.0): 0.5854
TF-IDF+CNB(α=0.001): 0.6436
TF-IDF+CNB(α=0.01): 0.6689
TF-IDF+CNB(α=0.1): 0.6954
TF-IDF+CNB(α=1.0): 0.6930
TF-IDF Filtro+MNB(α=0.001): 0.6603
TF-IDF Filtro+MNB(α=0.01): 0.6801
TF-IDF Filtro+MNB(α=0.1): 0.6798
TF-IDF Filtro+MNB(α=1.0): 0.6512
TF-IDF Filtro+CNB(α=0.001): 0.6651
TF-IDF Filtro+CNB(α=0.01): 0.6729
TF-IDF Filtro+CNB(α=0.1): 0.6887
TF-IDF Filtro+CNB(α=1.0): 0.6943
TF-IDF Sublinear+MNB(α=0.001): 0.6618
TF-IDF Sublinear+MNB(α=0.01): 0.6794
TF-IDF Sublinear+MNB(α=0.1): 0.6715
TF-IDF Sublinear+MNB(α=1.0): 0.6391
TF-IDF Sublinear+CNB(α=0.001): 0.6319
TF-IDF Sublinear+CNB(α=0.01): 0.6613
TF-IDF Sublinear+CNB(α=0.1): 0.6900
TF-IDF Sublinear+CNB(α=1.0): 0.6920
TF-IDF MaxFeat+MNB(α=0.001): 0.6339
TF-IDF MaxFeat+MNB(α=0.01): 0.6510
TF-IDF MaxFeat+MNB(α=0.1): 0.6625
TF-IDF MaxFeat+MNB(α=1.0): 0.6462
TF-IDF MaxFeat+CNB(α=0.001): 0.6549
TF-IDF MaxFeat+CNB(α=0.01): 0.6548
TF-IDF MaxFe

Algunas observaciones

- Mucho mejor el resultado optimizado con F1-Score Macro: 0.6954

- ComplementNB con TF-IDF y α=0.1 es el mejor
- En general, ComplementNB es mejor que MultinomialNB
- TF-IDF es claramente superior a Count Vectorizer
- MultinomialNB puede ganar con alphas muy pequeños
- stop words permite que las palabras distintivas de cada clase tengan mayor peso en la clasificación


In [55]:
# 4. Transponer la matriz para obtener vectores de palabras
X_words = X_train.T 

# Palabras elegidas manualmente para estudio
target_words = ['music', 'soccer', 'technology', 'jesus', 'sick']

for word in target_words:
    if word in tfidfvect.vocabulary_:
        word_idx = tfidfvect.vocabulary_[word]
        word_sim = cosine_similarity(X_words[word_idx], X_words)[0]
        # top 5 asumimos que el primer elemento es la palabra en sí
        top_indices = np.argsort(word_sim)[::-1][1:6]
        
        print(f"\nPalabra: '{word}'")
        for i in top_indices:
            print(f" - {idx2word[i]} (Sim: {word_sim[i]:.3f})")
    else:
        print(f"La palabra '{word}' no está en el vocabulario.")


Palabra: 'music'
 - inconveniences (Sim: 0.415)
 - colect (Sim: 0.415)
 - posses (Sim: 0.415)
 - deaf (Sim: 0.408)
 - competencies (Sim: 0.402)

Palabra: 'soccer'
 - philc (Sim: 0.549)
 - mattel (Sim: 0.328)
 - influx (Sim: 0.318)
 - borshevsky (Sim: 0.299)
 - blunted (Sim: 0.299)

Palabra: 'technology'
 - blatently (Sim: 0.310)
 - lecturing (Sim: 0.310)
 - ellul (Sim: 0.310)
 - christan (Sim: 0.310)
 - toffler (Sim: 0.310)

Palabra: 'jesus'
 - christ (Sim: 0.304)
 - god (Sim: 0.269)
 - kingdom (Sim: 0.213)
 - mat (Sim: 0.197)
 - bible (Sim: 0.195)

Palabra: 'sick'
 - phrase (Sim: 0.262)
 - magnavox (Sim: 0.252)
 - babies (Sim: 0.232)
 - flyback (Sim: 0.231)
 - liked (Sim: 0.216)


Interpretación

Al transponer la matriz se está representando cada palabra en el "contexto" de los documentos. Y los resultados muestran similitud en ocurrencia documental, no semántica.